# Exploración MinIO Lakehouse

Notebook para:
- leer `users.csv`, `users.parquet`, `users.avro`, `users.json` desde MinIO,
- comparar tamaños,
- visualizar esquemas,
- añadir tus datos personales a `users.parquet`,
- leer columnas específicas y aplicar filtro pushdown,
- fusionar `data.parquet` y `users.parquet` en `merged_data.parquet`.


In [ ]:
!pip install -q boto3 pandas pyarrow fastavro


In [ ]:
import os
import json
from io import BytesIO

import boto3
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from fastavro import reader as avro_reader

ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
BUCKET = "datalake"

endpoints = [
    os.getenv("MINIO_ENDPOINT"),
    "http://minio:9000",
    "http://localhost:9000",
]

endpoints = [e for e in endpoints if e]
s3 = None
for ep in endpoints:
    try:
        tmp = boto3.client(
            "s3",
            endpoint_url=ep,
            aws_access_key_id=ACCESS_KEY,
            aws_secret_access_key=SECRET_KEY,
        )
        tmp.list_buckets()
        s3 = tmp
        print(f"Conectado a MinIO en: {ep}")
        break
    except Exception as e:
        print(f"No conecta en {ep}: {e}")

if s3 is None:
    raise RuntimeError(f"No se pudo conectar a MinIO. Endpoints probados: {endpoints}")


In [ ]:
resp = s3.list_objects_v2(Bucket=BUCKET)
print("Objetos en bucket:", resp.get("KeyCount", 0))
for o in sorted(resp.get("Contents", []), key=lambda x: x["Key"]):
    print(f"- {o['Key']} ({o['Size']} bytes)")


In [ ]:
csv_obj = s3.get_object(Bucket=BUCKET, Key="users.csv")
df_csv = pd.read_csv(BytesIO(csv_obj["Body"].read()))
df_csv.head()


In [ ]:
parquet_obj = s3.get_object(Bucket=BUCKET, Key="users.parquet")
df_parquet = pq.read_table(BytesIO(parquet_obj["Body"].read())).to_pandas()
df_parquet.head()


In [ ]:
avro_obj = s3.get_object(Bucket=BUCKET, Key="users.avro")
with BytesIO(avro_obj["Body"].read()) as avro_buffer:
    av = avro_reader(avro_buffer)
    avro_records = [r for r in av]

df_avro = pd.DataFrame(avro_records)
df_avro.head()


In [ ]:
json_obj = s3.get_object(Bucket=BUCKET, Key="users.json")
json_data = json.loads(json_obj["Body"].read())
df_json = pd.DataFrame(json_data["users"])
df_json.head()


In [ ]:
print("Tamaños de archivo")
for key in ["users.csv", "users.parquet", "users.avro", "users.json", "data.parquet"]:
    try:
        meta = s3.head_object(Bucket=BUCKET, Key=key)
        print(f"{key:15} -> {meta['ContentLength']} bytes")
    except Exception as e:
        print(f"{key:15} -> no disponible ({e})")


In [ ]:
# Esquema Parquet
p_bytes = s3.get_object(Bucket=BUCKET, Key="users.parquet")["Body"].read()
pfile = pq.ParquetFile(BytesIO(p_bytes))
print("Schema Parquet:")
print(pfile.schema)

# Esquema Avro
a_bytes = s3.get_object(Bucket=BUCKET, Key="users.avro")["Body"].read()
with BytesIO(a_bytes) as b:
    av = avro_reader(b)
    print("\nSchema Avro:")
    print(av.writer_schema)


## Añade tus datos personales a `users.parquet`

Edita el diccionario de `mi_fila` con tu información real (`@up.edu`) y ejecuta la celda.


In [ ]:
users_bytes = s3.get_object(Bucket=BUCKET, Key="users.parquet")["Body"].read()
users_df = pq.read_table(BytesIO(users_bytes)).to_pandas()

# TODO: reemplaza con tus datos
mi_fila = {
    "user_id": 9999,
    "name": "TU_NOMBRE",
    "email": "tu_correo@up.edu",
    "signup_date": pd.Timestamp("2026-03-10")
}

nuevo_df = pd.concat([users_df, pd.DataFrame([mi_fila])], ignore_index=True)
nueva_tabla = pa.Table.from_pandas(nuevo_df, preserve_index=False)

buf = BytesIO()
pq.write_table(nueva_tabla, buf)
s3.put_object(Bucket=BUCKET, Key="users.parquet", Body=buf.getvalue())

print("users.parquet actualizado")
nuevo_df.tail(5)


In [ ]:

data_bytes = s3.get_object(Bucket=BUCKET, Key="data.parquet")["Body"].read()
df_cols = pq.read_table(BytesIO(data_bytes), columns=["Name", "Age"]).to_pandas()
df_cols.head()


In [ ]:

data_bytes = s3.get_object(Bucket=BUCKET, Key="data.parquet")["Body"].read()
filtered_table = pq.read_table(BytesIO(data_bytes), filters=[("Age", ">", 30)])
df_filtered = filtered_table.to_pandas()
df_filtered


In [ ]:

users_df = pq.read_table(BytesIO(s3.get_object(Bucket=BUCKET, Key="users.parquet")["Body"].read())).to_pandas()
data_df = pq.read_table(BytesIO(s3.get_object(Bucket=BUCKET, Key="data.parquet")["Body"].read())).to_pandas()

if "signup_date" not in users_df.columns:
    users_df["signup_date"] = pd.NaT

users_norm = users_df[["user_id", "name", "email", "signup_date"]].copy()

from_data = pd.DataFrame({
    "user_id": pd.NA,
    "name": data_df.get("Name"),
    "email": pd.NA,
    "signup_date": pd.NaT,
})

merged_df = pd.concat([users_norm, from_data], ignore_index=True)
merged_table = pa.Table.from_pandas(merged_df, preserve_index=False)

mbuf = BytesIO()
pq.write_table(merged_table, mbuf)
s3.put_object(Bucket=BUCKET, Key="merged_data.parquet", Body=mbuf.getvalue())

print("merged_data.parquet subido")
merged_df.tail(10)


In [ ]:
for key in ["users.parquet", "data.parquet", "merged_data.parquet"]:
    meta = s3.head_object(Bucket=BUCKET, Key=key)
    print(f"{key:20} -> {meta['ContentLength']} bytes")
